# AI Agent (local, free — Ollama only)

This is your own extended version of the agent, kept as the base since you
already ran it against a real local Ollama model and it worked. Three things
changed here:

1. **Engine core swapped in for the merged `03_recommendation_engine.ipynb`**
   — skill canonicalization, the BAAI semantic model, `career_goal` /
   `learner_description` inputs, updated 0.40/0.45/0.15 hybrid weights, and
   rank + contribution output.
2. **`get_recommendations` and its tool schema now also accept `career_goal`
   and `learner_description`**, so the agent can pass richer context into
   the semantic signal instead of just a bare skill list.
3. **A grounding check was added after the model's final reply.** Your own
   test run of the old version shows why this was needed — see the note
   right before Step 5.

Everything else is yours, unchanged: the requirement that Ollama actually be
running (no scripted fallback), the fake-tool-call detector for models that
type out a tool call as text instead of using the real mechanism, per-tool
error handling, the two test scenarios, the sanity-check cell, and the
interactive `chat()` loop.


## Step 1 — Setup, data, and the recommendation engine core

In [27]:
import ast
import json

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.width", 160)

user_profiles = pd.read_csv("data/recommendation_ready/user_profiles.csv")
course_profiles = pd.read_csv("data/recommendation_ready/course_profiles.csv")


def parse_list_cell(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return [s.strip() for s in str(value).split(",") if s.strip()]


course_profiles["skills_list"] = course_profiles["skills_list"].apply(parse_list_cell)
user_profiles["skills"] = user_profiles["skills"].apply(parse_list_cell)

print("Loaded", len(user_profiles), "users and", len(course_profiles), "courses.")

Loaded 54933 users and 404 courses.


In [28]:
# --- Engine core, matching 03_recommendation_engine.ipynb ---

def normalize_skill(skill):
    if pd.isna(skill):
        return None
    skill = str(skill).lower().strip()
    skill = skill.replace("&", " and ")
    skill = skill.replace("-", " ")
    skill = " ".join(skill.split())
    return skill if skill else None


# Known skill aliases / equivalent names - extend as you find more
skill_aliases = {
    "python": "python",
    "python programming": "python",
    "ml": "machine learning",
    "machine learning": "machine learning",
    "machine learning algorithms": "machine learning",
    "sql": "sql",
    "statistics": "statistics",
    "general statistics": "statistics",
    "probability and statistics": "statistics",
    "tableau": "tableau",
    "tableau software": "tableau",
}


def canonicalize_skill(skill):
    normalized = normalize_skill(skill)
    if normalized is None:
        return None
    return skill_aliases.get(normalized, normalized)


course_profiles["skills_canonical"] = course_profiles["skills_list"].apply(
    lambda skills: [canonicalize_skill(s) for s in skills]
)


def compute_skill_gap(current_skills, target_skills):
    current_canonical = {canonicalize_skill(s) for s in current_skills}
    target_canonical = [canonicalize_skill(s) for s in target_skills]
    target_canonical = [s for s in target_canonical if s]
    return [s for s in target_canonical if s not in current_canonical]


def compute_weighted_skill_gap_coverage(course_profiles, skill_gap, skill_weights=None):
    if skill_weights is None:
        skill_weights = {}
    weights = {s: skill_weights.get(s, 1.0) for s in skill_gap}
    total_weight = sum(weights.values())
    if not skill_gap or total_weight == 0:
        return pd.Series(0.0, index=course_profiles.index)

    def coverage_for_course(course_skills):
        course_skill_set = set(course_skills)
        covered_weight = sum(w for s, w in weights.items() if s in course_skill_set)
        return covered_weight / total_weight

    return course_profiles["skills_canonical"].apply(coverage_for_course)


SKILL_SEPARATOR = " ||| "


def compute_tfidf_similarity(course_profiles, skill_gap):
    if not skill_gap:
        return pd.Series(0.0, index=course_profiles.index)

    def skill_tokenizer(text):
        return text.split(SKILL_SEPARATOR)

    course_profiles["tfidf_skill_text"] = course_profiles["skills_canonical"].apply(
        lambda skills: SKILL_SEPARATOR.join(skills)
    )
    vectorizer = TfidfVectorizer(tokenizer=skill_tokenizer, preprocessor=None,
                                  token_pattern=None, lowercase=False)
    course_matrix = vectorizer.fit_transform(course_profiles["tfidf_skill_text"])
    gap_vector = vectorizer.transform([SKILL_SEPARATOR.join(skill_gap)])
    sims = cosine_similarity(gap_vector, course_matrix).flatten()
    return pd.Series(sims, index=course_profiles.index)


def build_course_semantic_text(row):
    return (f"Title: {row['title']}. Skills: {', '.join(row['skills_canonical'])}. "
            f"Description: {row.get('course_description_clean', '')}")


def compute_semantic_similarity(course_profiles, skill_gap, model, career_goal="", learner_description=""):
    if model is None or not skill_gap:
        return None
    if "semantic_text" not in course_profiles.columns:
        course_profiles["semantic_text"] = course_profiles.apply(build_course_semantic_text, axis=1)
    learner_semantic_text = (f"Career Goal: {career_goal}. Required Skills: {', '.join(skill_gap)}. "
                              f"Description: {learner_description}")
    course_embeddings = model.encode(course_profiles["semantic_text"].tolist(), normalize_embeddings=True)
    learner_embedding = model.encode([learner_semantic_text], normalize_embeddings=True)
    sims = cosine_similarity(learner_embedding, course_embeddings).flatten()
    return pd.Series(sims, index=course_profiles.index)


def filter_candidates(scores_df, semantic_available):
    if semantic_available:
        mask = ((scores_df["skill_gap_coverage"] > 0) | (scores_df["tfidf_similarity"] > 0)
                | (scores_df["semantic_similarity"] >= 0.40))
    else:
        mask = (scores_df["skill_gap_coverage"] > 0) | (scores_df["tfidf_similarity"] > 0)
    return scores_df[mask].copy()


def normalize_scores(candidates, columns):
    scaler = MinMaxScaler()
    normed = scaler.fit_transform(candidates[columns])
    for i, col in enumerate(columns):
        candidates[f"{col}_norm"] = normed[:, i]
    return candidates


def compute_hybrid_score(candidates, semantic_available,
                          coverage_weight=0.40, semantic_weight=0.45, tfidf_weight=0.15):
    if semantic_available:
        candidates = normalize_scores(candidates, ["skill_gap_coverage", "tfidf_similarity", "semantic_similarity"])
        candidates["hybrid_score"] = (coverage_weight * candidates["skill_gap_coverage_norm"]
                                       + semantic_weight * candidates["semantic_similarity_norm"]
                                       + tfidf_weight * candidates["tfidf_similarity_norm"])
    else:
        candidates = normalize_scores(candidates, ["skill_gap_coverage", "tfidf_similarity"])
        remaining = coverage_weight + tfidf_weight
        candidates["hybrid_score"] = ((coverage_weight / remaining) * candidates["skill_gap_coverage_norm"]
                                       + (tfidf_weight / remaining) * candidates["tfidf_similarity_norm"])
    return candidates


def apply_metadata_adjustment(candidates, course_profiles, metadata_weight=0.10):
    ratings = course_profiles.loc[candidates.index, "ratings"]
    ratings_filled = ratings.fillna(ratings.median())
    candidates["rating_norm"] = MinMaxScaler().fit_transform(ratings_filled.to_frame())[:, 0]
    candidates["final_score"] = ((1 - metadata_weight) * candidates["hybrid_score"]
                                  + metadata_weight * candidates["rating_norm"])
    return candidates


def diversity_filter(ranked_candidates, course_profiles, top_k=10, max_skill_overlap=0.6):
    selected, selected_skill_sets = [], []
    for idx, row in ranked_candidates.iterrows():
        skills = set(course_profiles.loc[idx, "skills_canonical"])
        too_similar = False
        for existing in selected_skill_sets:
            if not skills or not existing:
                continue
            if len(skills & existing) / len(skills | existing) >= max_skill_overlap:
                too_similar = True
                break
        if not too_similar:
            selected.append(idx)
            selected_skill_sets.append(skills)
        if len(selected) >= top_k:
            break
    return ranked_candidates.loc[selected]


def generate_recommendations(current_skills, target_skills, skill_weights=None,
                              career_goal="", learner_description="", top_k=8,
                              semantic_model=None, metadata_weight=0.10):
    skill_gap = compute_skill_gap(current_skills, target_skills)

    scores = pd.DataFrame(index=course_profiles.index)
    scores["skill_gap_coverage"] = compute_weighted_skill_gap_coverage(course_profiles, skill_gap, skill_weights)
    scores["tfidf_similarity"] = compute_tfidf_similarity(course_profiles, skill_gap)

    semantic_scores = compute_semantic_similarity(course_profiles, skill_gap, semantic_model,
                                                   career_goal, learner_description)
    semantic_available = semantic_scores is not None
    if semantic_available:
        scores["semantic_similarity"] = semantic_scores

    candidates = filter_candidates(scores, semantic_available)
    candidates = compute_hybrid_score(candidates, semantic_available)
    candidates = apply_metadata_adjustment(candidates, course_profiles, metadata_weight)

    ranked = candidates.sort_values("final_score", ascending=False)
    final = diversity_filter(ranked, course_profiles, top_k=top_k)

    result = course_profiles.loc[
        final.index, ["course_id", "title", "organization", "difficulty", "ratings", "course_url"]
    ].copy()
    result["final_score"] = final["final_score"]
    result = result.sort_values("final_score", ascending=False).reset_index(drop=True)
    result.insert(0, "rank", result.index + 1)
    return skill_gap, result


try:
    from sentence_transformers import SentenceTransformer
    semantic_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
except Exception:
    semantic_model = None

print("Semantic similarity available:", semantic_model is not None)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Semantic similarity available: True


## Step 2 — Tools the agent can call

Same three tools as before. `get_recommendations` now also accepts optional
`career_goal` and `learner_description`, which feed the richer semantic
signal in the merged engine.

In [29]:
NEW_USER_SESSIONS = {}


def lookup_existing_user(person_id):
    row = user_profiles.loc[user_profiles["person_id"] == int(person_id)]
    if row.empty:
        return {"found": False, "message": f"No user found with person_id={person_id}."}
    row = row.iloc[0]
    return {
        "found": True,
        "person_id": int(row["person_id"]),
        "career_context": row["career_context"],
        "current_skills": row["skills"],
    }


def save_new_user_profile(current_skills, career_context=""):
    session_id = f"new_user_{len(NEW_USER_SESSIONS) + 1}"
    normalized = [canonicalize_skill(s) for s in current_skills]
    normalized = [s for s in normalized if s]
    NEW_USER_SESSIONS[session_id] = {"career_context": career_context, "current_skills": normalized}
    return {"session_id": session_id, "current_skills": normalized, "career_context": career_context}


def get_recommendations(current_skills, target_skills, skill_weights=None,
                         career_goal="", learner_description="", top_k=8):
    skill_weights = skill_weights or {}
    skill_gap, recs = generate_recommendations(
        current_skills, target_skills, skill_weights=skill_weights,
        career_goal=career_goal, learner_description=learner_description,
        top_k=top_k, semantic_model=semantic_model,
    )
    return {"skill_gap": skill_gap, "recommended_courses": recs.to_dict(orient="records")}


TOOL_IMPLEMENTATIONS = {
    "lookup_existing_user": lookup_existing_user,
    "save_new_user_profile": save_new_user_profile,
    "get_recommendations": get_recommendations,
}

## Step 3 — Tool schemas and the system prompt

In [30]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_existing_user",
            "description": "Look up a learner who already exists in the company dataset, by person_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "person_id": {"type": "integer", "description": "The learner's person_id."}
                },
                "required": ["person_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_new_user_profile",
            "description": "Register a learner who is not in the company dataset, using skills and "
                            "context gathered from the conversation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "current_skills": {"type": "array", "items": {"type": "string"},
                                        "description": "Normalized list of the learner's current skills."},
                    "career_context": {"type": "string", "description": "The learner's current role or job title."},
                },
                "required": ["current_skills"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_recommendations",
            "description": "Run the recommendation engine and return a ranked, diversified list of "
                            "courses that close the gap between current_skills and target_skills.",
            "parameters": {
                "type": "object",
                "properties": {
                    "current_skills": {"type": "array", "items": {"type": "string"}},
                    "target_skills": {"type": "array", "items": {"type": "string"}},
                    "skill_weights": {
                        "type": "object",
                        "description": "Optional {skill: importance_weight} map. Higher weight = "
                                        "more important to the learner's stated goal and timeframe. "
                                        "Omit or leave a skill out for equal (1.0) weight.",
                    },
                    "career_goal": {"type": "string",
                                     "description": "The learner's stated target role, e.g. "
                                                     "\"Machine Learning Engineer\"."},
                    "learner_description": {"type": "string",
                                             "description": "A short free-text description of what the "
                                                             "learner is trying to achieve and why - used "
                                                             "to improve semantic matching."},
                    "top_k": {"type": "integer", "description": "How many courses to return."},
                },
                "required": ["current_skills", "target_skills"],
            },
        },
    },
]

In [31]:
SYSTEM_PROMPT = """You are a Learning & Development advisor agent for a corporate training
platform. Your job is to turn a conversation with an employee into a personalised course
learning path.

For every learner, first work out which case you're in:
- EXISTING USER: they give you a person_id, or say they already work here / are in the system.
  Call lookup_existing_user with that person_id. If it's not found, treat them as a new user
  instead and say so.
- NEW USER: no person_id, or lookup_existing_user came back not found. Build their profile
  from the conversation instead of guessing - ask about their current role and skills if they
  haven't already said enough for you to be confident, then call save_new_user_profile.

Once you know their current_skills:
1. Work out target_skills for their stated career goal, using your own knowledge of what that
   role typically requires. Don't ask for more detail before proceeding - make reasonable,
   standard assumptions instead (e.g. a general set of skills for the stated role, and a
   default timeframe of around a year if none was given).
2. Assign skill_weights to the target skills based on the stated goal and timeframe. A tighter
   timeframe should weight foundational, highest-leverage skills more heavily than nice-to-have
   ones. Equal weights are fine if you have no real basis to differentiate.
3. Write a one or two sentence learner_description of what they're trying to achieve, and pass
   it along with career_goal into get_recommendations - this is what makes the semantic matching
   good, not just a bag of skill keywords.
4. Call get_recommendations with current_skills, target_skills, skill_weights, career_goal, and
   learner_description, using the real tool-calling mechanism - never write out a tool call as
   text in your reply. You must base your answer only on the courses this tool actually
   returns - never invent, assume, or recall a course, a course URL, or a course title from
   general knowledge, even a well-known one. If you have not received a real result from
   get_recommendations yet, you are not finished - do not write a learning path yet.
5. Turn the ranked course list into a short, sequenced learning path in plain language - for
   every course you mention, use its exact title and include its course_url from the tool
   result right after the title (e.g. "Course Title (https://...)"). Briefly explain why each
   fits their goal and timeframe; don't just repeat the raw scores, and don't recommend a
   course that isn't in the tool result.
6. After the learning path, add one short closing note offering to personalise further.
   FIRST, re-read what the learner actually said. Build a short list, in your head, of what they
   already told you (role/goal, timeframe, any current skill level or subfield they mentioned).
   Then only invite the details NOT on that list. Concretely: if they said "6 months" or gave
   any timeframe at all, do not mention timeframe in this note, under any wording, in any
   form - not "your target timeframe", not "how much time you have", nothing about time. Same
   for target role: if they already named one, don't ask for it again. Only offer to hear about
   things that are genuinely still unknown, such as their current skill level with a specific
   tool, or which subfield interests them, if those truly weren't mentioned. If everything
   relevant was already given, this note can be one short sentence, or skipped entirely - do
   not pad it out with a repeated question just to have something to say.

Never narrate your own process to the learner. Don't say things like "I will now call...",
"Let me look that up", "Now I will work out your target skills", or describe your reasoning
about weights as a play-by-play. Do the reasoning and the tool calls silently, and only send
the learner the finished result: the closing note in step 6 is the only place you mention what
you assumed.

DO NOT ask the learner a clarifying question before producing a recommendation - always give
them a complete, real recommendation first using reasonable assumptions, and offer to refine it
afterward instead. The only exception is a NEW USER whose message doesn't give you enough to
even call save_new_user_profile (no discernible current skills or role at all) - in that one
case, ask a single short question to get the minimum you need, then proceed as above.
"""


## Step 4 — The tool-calling loop against Ollama

Same loop as your version, plus one addition explained below.

**Why a grounding check was added.** Your own test run of the previous
version (visible in your uploaded notebook's output) shows the model
producing course titles like "Python for Data Science", "Deep Learning with
TensorFlow", and "Linear Algebra and Calculus" with literal `https://...`
placeholder URLs, and in Scenario B, `https://example.com/...` URLs. None of
those are real entries in `course_profiles.csv` — the model wrote a
plausible-sounding answer instead of using the tool's actual result, even
though it did call `lookup_existing_user` for real and even though the
system prompt explicitly says not to do this.

This is a known failure mode for smaller local models: they can call a tool
correctly and still ignore its result when writing the final free-text
answer. Prompting alone didn't fix it, so this adds a code-level check after
the model's final reply: pull the titles from the actual last
`get_recommendations` result, and if the reply doesn't mention a real
majority of them, throw the model's prose away and render the list directly
from the tool's JSON instead. Same philosophy as your `extract_faked_tool_call`
check, just applied to the final answer instead of a mid-conversation tool
call.

In [32]:
import re

OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "llama3.1"  # any tool-calling capable model you have pulled


def ollama_is_available():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except requests.exceptions.RequestException:
        return False


def call_ollama_chat(messages, tools=None, model=OLLAMA_MODEL):
    payload = {"model": model, "messages": messages, "stream": False}
    if tools:
        payload["tools"] = tools
    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]


def extract_faked_tool_call(content):
    """Small models sometimes skip the real tool-calling mechanism and instead
    type out something that LOOKS like a tool call as plain text, e.g.
    {"name": "get_recommendations", "parameters": {...}} - then continue on to
    invent fictional results after it. Detect that pattern so we can run the
    REAL tool instead of trusting whatever the model made up next."""
    if not content:
        return None
    match = re.search(r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"parameters"\s*:\s*(\{.*)',
                       content, re.DOTALL)
    if not match:
        return None
    name = match.group(1)
    if name not in TOOL_IMPLEMENTATIONS:
        return None
    rest = match.group(2)
    depth, params_str = 0, None
    for i, ch in enumerate(rest):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                params_str = rest[: i + 1]
                break
    if params_str is None:
        return None
    try:
        arguments = json.loads(params_str)
    except json.JSONDecodeError:
        return None
    return {"name": name, "arguments": arguments}

In [33]:
def extract_last_recommendation_data(messages):
    """Find the most recent get_recommendations tool result in this
    conversation and return its full {"skill_gap": [...], "courses": [...]},
    or None if no recommendation has happened yet."""
    for m in reversed(messages):
        if m.get("role") != "tool":
            continue
        try:
            data = json.loads(m["content"])
        except (json.JSONDecodeError, TypeError):
            continue
        courses = data.get("recommended_courses")
        if courses:
            return {"skill_gap": data.get("skill_gap", []), "courses": courses}
    return None


def is_reply_grounded(reply_text, courses):
    """Cheap proxy for 'did the model actually use the tool result, or did
    it make courses up': at least half of the real titles should appear
    verbatim in the reply. Only meaningful when courses actually exist -
    see looks_like_a_recommendation() for the "no real data at all" case,
    which this function does not handle."""
    if not courses:
        return True
    reply_lower = (reply_text or "").lower()
    mentioned = sum(1 for c in courses if c.get("title") and c["title"].lower() in reply_lower)
    return mentioned >= max(1, len(courses) // 2)


def looks_like_a_recommendation(reply_text):
    """Heuristic: does this reply look like it's presenting course
    recommendations at all (a link, or a numbered list of 2+ items)?
    Used only when NO real get_recommendations call has happened yet in
    this conversation - if the model is presenting something that looks
    like a course list with zero real data behind it, that is a
    fabrication by definition, not an edge case to shrug off."""
    if not reply_text:
        return False
    has_link = "http" in reply_text.lower()
    numbered_items = len(re.findall(r"(?m)^\s*\d+[.)]\s", reply_text))
    return has_link or numbered_items >= 2


def course_skill_match(course_id, skill_gap):
    """Real, deterministic reason a course was recommended: which of the
    learner's skill-gap skills it actually covers, looked up straight from
    course_profiles - no LLM involved, so this is trustworthy even in the
    exact situation where the model's own explanation isn't."""
    row = course_profiles.loc[course_profiles["course_id"] == course_id]
    if row.empty:
        return []
    skills_canonical = set(row.iloc[0]["skills_canonical"])
    return sorted(skills_canonical & set(skill_gap))


def humanize_list(items):
    """['python', 'sql'] -> 'python and sql'; ['a','b','c'] -> 'a, b, and c'."""
    items = list(items)
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"


DIFFICULTY_BLURBS = {
    "beginner": "it's beginner-friendly, so you don't need prior experience to start",
    "intermediate": "it's pitched at an intermediate level, so it helps to have the basics down first",
    "advanced": "it's an advanced course, built for people who already have some hands-on experience",
    "mixed": "it covers a mix of levels, so there's something in it wherever you're starting from",
}


def build_fallback_reply(real_data):
    """Deterministic rendering straight from the tool's real JSON result,
    used only when the model's free-text reply failed the grounding check.
    Each course gets a real, data-backed explanation written as plain,
    conversational sentences - not raw skill tags or a spec sheet, since
    this is meant to be readable by someone new to the field. Nothing here
    mentions that this is a fallback; that's an implementation detail, not
    something the learner needs to know."""
    skill_gap = real_data.get("skill_gap", [])
    courses = real_data.get("courses", [])

    lines = ["Here's a learning path built around the skills you're still missing:", ""]
    for course in courses:
        title = course.get("title", "Untitled course")
        url = course.get("course_url")
        rating = course.get("ratings")
        difficulty = (course.get("difficulty") or "").strip().lower()

        line = f"{course.get('rank', '?')}. {title}"
        if url:
            line += f" ({url})"
        lines.append(line)

        covered = course_skill_match(course.get("course_id"), skill_gap)
        if covered:
            sentence = f"This one covers {humanize_list(covered)} - skills that are on your list to learn"
        else:
            sentence = "This one rounds out your path toward the goal you're working on"

        extra_bits = []
        blurb = DIFFICULTY_BLURBS.get(difficulty)
        if blurb:
            extra_bits.append(blurb)
        if rating:
            extra_bits.append(f"learners have rated it {rating} out of 5")

        if extra_bits:
            sentence += " - " + "; ".join(extra_bits) + "."
        else:
            sentence += "."

        lines.append("   " + sentence)
        lines.append("")

    return "\n".join(lines).strip()


NO_REAL_DATA_REMINDER = (
    "You have NOT called get_recommendations yet in this conversation, so you have no real "
    "course data - none. The course titles, descriptions, and links in your last reply were "
    "invented, not real. Do not write, repeat, or reference any of them again. Call the "
    "get_recommendations tool for real right now, using the current_skills, target_skills, and "
    "skill_weights you already worked out. Only write your learning-path reply after you "
    "receive its actual result."
)


def run_agent_turn(messages, tools=TOOLS, model=OLLAMA_MODEL, max_tool_iterations=6):
    """One user turn: keep calling the model and executing tool calls
    until it returns a plain-text reply. Returns (final_reply, updated_messages)."""
    for _ in range(max_tool_iterations):
        assistant_message = call_ollama_chat(messages, tools=tools, model=model)
        tool_calls = assistant_message.get("tool_calls")

        if tool_calls:
            messages.append(assistant_message)
        else:
            reply_text = assistant_message.get("content", "")
            faked = extract_faked_tool_call(reply_text)

            if faked:
                assistant_message = {"role": "assistant", "content": None,
                                      "tool_calls": [{"function": faked}]}
                tool_calls = assistant_message["tool_calls"]
                messages.append(assistant_message)
            else:
                messages.append(assistant_message)
                real_data = extract_last_recommendation_data(messages)

                if real_data is not None:
                    # A real get_recommendations call happened at some point in this
                    # conversation - check the model's summary actually reflects it.
                    if not is_reply_grounded(reply_text, real_data["courses"]):
                        print("[grounding check] Reply didn't match the real tool result - "
                              "rendering the data-backed fallback instead.")
                        reply_text = build_fallback_reply(real_data)
                        messages[-1] = {"role": "assistant", "content": reply_text}
                    return reply_text, messages

                elif looks_like_a_recommendation(reply_text):
                    # Worst case: no real tool call has EVER happened in this
                    # conversation, yet the model is presenting what looks like a
                    # course recommendation anyway. There is no real data to fall
                    # back to - force a retry instead of returning fabricated content.
                    messages.append({"role": "user", "content": NO_REAL_DATA_REMINDER})
                    continue

                else:
                    # A genuine non-recommendation reply (e.g. a clarifying question
                    # for a brand-new user) - nothing to ground, nothing fabricated.
                    return reply_text, messages

        # tool_calls is guaranteed truthy here (either a real native call, or a
        # faked-as-text call we just converted into the real mechanism above).
        for call in tool_calls:
            name = call["function"]["name"]
            arguments = call["function"]["arguments"]
            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            tool_fn = TOOL_IMPLEMENTATIONS.get(name)
            if tool_fn is None:
                result = {"error": f"Unknown tool {name}"}
            else:
                try:
                    result = tool_fn(**arguments)
                except TypeError as e:
                    result = {"error": f"Bad arguments for {name}: {e}. Check the tool's "
                                        "schema and retry with the correct argument names."}
                except Exception as e:
                    result = {"error": f"{name} raised {type(e).__name__}: {e}"}

            messages.append({"role": "tool", "content": json.dumps(result, default=str)})

    return "(stopped after too many tool calls without a final answer)", messages


## Step 4b — Unit-testing the grounding check

This sandbox can't reach your Ollama server, so the two live scenarios
below need to run on your machine. What can be tested here is the
grounding logic itself, against a real `get_recommendations` result -
including the exact bug that got through last time: a reply presenting
fake courses when **no real tool call had happened at all yet**, which
the first version of this check let through because it only knew how to
compare against real data, not notice that none existed.


In [34]:
real_result = get_recommendations(
    current_skills=["database administration", "sql", "sql server"],
    target_skills=["python", "machine learning", "data science"],
    career_goal="Machine Learning Engineer",
    learner_description="Move from database administration into ML engineering.",
    top_k=5,
)
real_data = {"skill_gap": real_result["skill_gap"], "courses": real_result["recommended_courses"]}
print("Real course titles from the engine:")
for c in real_data["courses"]:
    print(" -", c["title"])

hallucinated_reply = (
    "1. Python for Data Science (https://...)\n"
    "2. Machine Learning Fundamentals (https://...)\n"
    "3. Deep Learning with TensorFlow (https://...)"
)
grounded_reply = "Here is your path: " + ", ".join(c["title"] for c in real_data["courses"])

print("\n--- Case 1: real data exists, reply strayed from it ---")
print("Hallucinated reply flagged as grounded?", is_reply_grounded(hallucinated_reply, real_data["courses"]))
print("Real reply flagged as grounded?        ", is_reply_grounded(grounded_reply, real_data["courses"]))
print("\nFallback rendering if grounding fails (now with real, data-backed explanations,")
print("no debug note visible to the learner):\n")
print(build_fallback_reply(real_data))

print("\n--- Case 2: the actual bug - NO real tool call ever happened ---")
print("This is exactly the shape of the reply that got through before the fix:")
print(hallucinated_reply)
print("\nlooks_like_a_recommendation(...)  ->", looks_like_a_recommendation(hallucinated_reply))
print("(True means run_agent_turn will now force a retry instead of returning this)")

plain_clarifying_question = "Could you tell me a bit about your current role?"
print("\nA genuine clarifying question should NOT be flagged:")
print(plain_clarifying_question)
print("looks_like_a_recommendation(...)  ->", looks_like_a_recommendation(plain_clarifying_question))


Real course titles from the engine:
 - IBM AI Engineering
 - Introduction to Data Science
 - IBM Data Engineering
 - IBM Applied AI
 - IBM Data Analyst

--- Case 1: real data exists, reply strayed from it ---
Hallucinated reply flagged as grounded? False
Real reply flagged as grounded?         True

Fallback rendering if grounding fails (now with real, data-backed explanations,
no debug note visible to the learner):

Here's a learning path built around the skills you're still missing:

1. IBM AI Engineering (https://www.coursera.org/professional-certificates/ai-engineer)
   This one covers data science, machine learning, and python - skills that are on your list to learn - it's pitched at an intermediate level, so it helps to have the basics down first; learners have rated it 4.6 out of 5.

2. Introduction to Data Science (https://www.coursera.org/specializations/introduction-data-science)
   This one covers data science, machine learning, and python - skills that are on your list to l

## Step 5 — Run it

In [35]:
def run_conversation(label, user_messages):
    """Send each user message through run_agent_turn in sequence, against the
    real local Ollama server, printing the agent's replies as it goes."""
    print("=" * 70)
    print(label)
    print("=" * 70)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_text in user_messages:
        print(f"\nUSER: {user_text}")
        messages.append({"role": "user", "content": user_text})
        reply, messages = run_agent_turn(messages)
        print(f"\nAGENT: {reply}")

    return messages

In [36]:
if not ollama_is_available():
    raise RuntimeError(
        "Ollama not reachable at http://localhost:11434 - start it with `ollama serve` "
        "and make sure a tool-calling model is pulled (e.g. `ollama pull llama3.1`)."
    )
print("Ollama detected on localhost:11434 - using the real model.")

Ollama detected on localhost:11434 - using the real model.


### Scenario A — existing user

In [37]:
messages_a = run_conversation(
    "SCENARIO A: existing user (person_id in the dataset)",
    ["I'm person_id 1 in the system. I want to become a machine learning engineer "
    "in the next 6 months, what should I learn?"],
)

SCENARIO A: existing user (person_id in the dataset)

USER: I'm person_id 1 in the system. I want to become a machine learning engineer in the next 6 months, what should I learn?

AGENT: Based on the tool's output, here's a short, sequenced learning path for you:

1. Machine Learning (https://www.coursera.org/specializations/machine-learning)
	* This course covers the fundamentals of machine learning, which is a crucial skill for machine learning engineers.
2. Machine Learning Engineering for Production (MLOps) (https://www.coursera.org/specializations/machine-learning-engineering-for-production-mlops)
	* This course covers the skills needed to work with machine learning models in production, including deployment and maintenance.
3. Improving Deep Neural Networks: Hyperparameter Tuning, Regularization and Optimization (https://www.coursera.org/learn/deep-neural-network)
	* This course covers the techniques needed to improve the performance of deep neural networks, including hyperparame

In [38]:
# Sanity check: did the agent actually call get_recommendations, and what
# did the real engine return? Run this right after Scenario A / Scenario B.
for m in messages_a:
    if m["role"] == "tool":
        print(m["content"][:1000])
        print("-" * 70)

{"found": true, "person_id": 1, "career_context": "database administrator", "current_skills": ["database administration", "database", "ms sql server", "ms sql server 2005", "sql server", "sql server 2005", "sql server 2008", "sql server 2008 r2", "sql server 2012", "sql", "sql queries", "stored procedures", "clustering", "backups", "t sql", "virtualization", "r2", "maintenance", "problem solving", "shipping"]}
----------------------------------------------------------------------
{"error": "get_recommendations raised AttributeError: 'list' object has no attribute 'get'"}
----------------------------------------------------------------------
{"skill_gap": ["python", "tensorflow", "pytorch", "convolutional neural networks", "recurrent neural networks", "long short term memory networks", "pandas", "numpy", "matplotlib", "accuracy", "precision", "recall"], "recommended_courses": [{"rank": 1, "course_id": "COURSE_0007", "title": "Machine Learning", "organization": "Multiple educators", "dif

### Scenario B — new user, not in the dataset

In [39]:
messages_b = run_conversation(
    "SCENARIO B: new user (not in the dataset)",
    ["Hi, I don't think I'm in your system. I work as a marketing coordinator and "
     "want to move into data analysis.",
     "Just the basics with SQL, mostly filtering and simple joins. Maybe within a year."],
)

SCENARIO B: new user (not in the dataset)

USER: Hi, I don't think I'm in your system. I work as a marketing coordinator and want to move into data analysis.

AGENT: Based on your goal of transitioning into data analysis, I recommend the following learning path:

1. "Introduction to Data Analytics" (https://www.coursera.org/learn/introduction-to-data-analytics) - As a marketing coordinator, you already have some experience with data analysis, but this course will help you build on that foundation and develop a deeper understanding of data analytics concepts. It's a beginner-friendly course that will provide you with a solid foundation in data analysis.

2. "Microsoft Power BI Data Analyst" (https://www.coursera.org/professional-certificates/microsoft-power-bi-data-analyst) - In this course, you'll learn how to use Power BI to effectively analyze and visualize data. This is a crucial skill for any data analyst, and it will help you present your findings to stakeholders in a clear and co

## Interactive chat

Run the cell below, then type your questions at the prompt and watch the
agent respond in real time. Type `quit`, `exit`, or an empty line to stop.

In [ ]:
def chat():
    """Interactive loop: type a message, see the agent's reply, repeat.
    Type 'quit', 'exit', or an empty line to stop."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    print("Chat with the L&D agent. Type 'quit' to stop.\n")

    while True:
        user_text = input("YOU: ").strip()
        if not user_text or user_text.lower() in ("quit", "exit"):
            print("Ending chat.")
            break

        messages.append({"role": "user", "content": user_text})
        reply, messages = run_agent_turn(messages)
        print(f"\nAGENT: {reply}\n")

    return messages


chat_messages = chat()